In [1]:
import os
import gc
import glob
import torch
import pandas as pd
from sdv.metadata import Metadata
from sdv.sequential import PARSynthesizer

# 1. Settings
WINDOW_SIZE = 50          # Rows per window
EPOCHS = 100
NUM_SYNTH_WINDOWS = 100   # Synthetic windows generated per dataset
MIN_WINDOWS = 5           # Skip datasets that yield fewer windows than this
ONLY = None               # e.g. ["xx80", "xx60"] to run just those files (stem names); None = all

SNAPSHOT_DIR = "./.snapshots"
SYNTH_DIR = "./data_features/synth"

# Bypass the cuDNN RNN crash while keeping GPU active
torch.backends.cudnn.enabled = False

os.makedirs(SNAPSHOT_DIR, exist_ok=True)
os.makedirs(SYNTH_DIR, exist_ok=True)


def make_windows(file_path):
    """Slice one CSV into non-overlapping windows, each with its own series_id."""
    df = pd.read_csv(file_path)
    windows = []
    window_id = 1
    for start in range(0, len(df) - WINDOW_SIZE + 1, WINDOW_SIZE):
        chunk = df.iloc[start : start + WINDOW_SIZE].copy()
        chunk["series_id"] = window_id
        # Reset Timer(ms) to start at 0.0 for each window
        chunk["Timer(ms)"] = chunk["Timer(ms)"] - chunk["Timer(ms)"].iloc[0]
        windows.append(chunk)
        window_id += 1
    if not windows:
        return None, 0
    return pd.concat(windows, ignore_index=True), window_id - 1


# 2. One dataset at a time: own windows, own metadata, own synthesizer
csv_files = sorted(glob.glob("./data_clean/*.csv"))

for file_path in csv_files:
    name = os.path.splitext(os.path.basename(file_path))[0]   # e.g. "xx80"
    if ONLY is not None and name not in ONLY:
        continue

    print(f"\n=== {name} ===")
    df, n_windows = make_windows(file_path)
    if df is None or n_windows < MIN_WINDOWS:
        print(f"Skipping {name}: only {n_windows} window(s) of {WINDOW_SIZE} rows.")
        continue
    print(f"Created {n_windows} windows of {WINDOW_SIZE} rows each ({len(df)} total rows).")

    # 3. Build and validate metadata (fresh for each dataset)
    table = name
    metadata = Metadata.detect_from_dataframes(data={table: df})
    metadata.update_column(table_name=table, column_name="series_id", sdtype="id")
    metadata.set_sequence_key(table_name=table, column_name="series_id")
    metadata.set_sequence_index(table_name=table, column_name="Timer(ms)")
    metadata.validate()

    # 4. Train and save, named after the dataset
    synthesizer = PARSynthesizer(metadata, epochs=EPOCHS, verbose=True)
    synthesizer.fit({table: df})
    synthesizer.save(f"{SNAPSHOT_DIR}/{name}_synthesizer.pkl")

    # 5. Generate and export synthetic windows
    synthetic = synthesizer.sample_sequences(
        num_sequences=NUM_SYNTH_WINDOWS,
        sequence_length=WINDOW_SIZE,
    )
    if isinstance(synthetic, dict):
        synthetic = synthetic[table]

    out_path = f"{SYNTH_DIR}/{name}+.csv"
    synthetic.to_csv(out_path, index=False)
    print(f"Saved {out_path}")

    # Free memory before the next dataset
    del synthesizer, synthetic, df, metadata
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nDone!")


=== Iisingintie_Acc1_m800_RDMC_2026-05-06_123750 ===
Created 1137 windows of 50 rows each (56850 total rows).


/home/akka/Documents/vscode_projects/SemesterProjectFall2026/.venv/lib/python3.12/site-packages/sdv/single_table/base.py:136: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
/home/akka/Documents/vscode_projects/SemesterProjectFall2026/.venv/lib/python3.12/site-packages/sdv/metadata/metadata.py:1226: UserWarning: The metadata lists columns in a different order than the data. This may result in the synthetic data having a different order.
Affected tables: 'Iisingintie_Acc1_m800_RDMC_2026-05-06_123750'.
  warnings.warn(
100%|██████████| 100/100 [01:27<00:00,  1.14it/s]


Saved ./data_features/synth/Iisingintie_Acc1_m800_RDMC_2026-05-06_123750+.csv

=== Iisingintie_Acc1_m801_RDMC_2026-05-06_124454 ===
Created 1509 windows of 50 rows each (75450 total rows).


/home/akka/Documents/vscode_projects/SemesterProjectFall2026/.venv/lib/python3.12/site-packages/sdv/single_table/base.py:136: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
/home/akka/Documents/vscode_projects/SemesterProjectFall2026/.venv/lib/python3.12/site-packages/sdv/metadata/metadata.py:1226: UserWarning: The metadata lists columns in a different order than the data. This may result in the synthetic data having a different order.
Affected tables: 'Iisingintie_Acc1_m801_RDMC_2026-05-06_124454'.
  warnings.warn(
100%|██████████| 100/100 [01:27<00:00,  1.14it/s]


Saved ./data_features/synth/Iisingintie_Acc1_m801_RDMC_2026-05-06_124454+.csv

=== Iisingintie_Acc1_m802_RDMC_2026-05-06_125211 ===
Created 2293 windows of 50 rows each (114650 total rows).


/home/akka/Documents/vscode_projects/SemesterProjectFall2026/.venv/lib/python3.12/site-packages/sdv/single_table/base.py:136: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
/home/akka/Documents/vscode_projects/SemesterProjectFall2026/.venv/lib/python3.12/site-packages/sdv/metadata/metadata.py:1226: UserWarning: The metadata lists columns in a different order than the data. This may result in the synthetic data having a different order.
Affected tables: 'Iisingintie_Acc1_m802_RDMC_2026-05-06_125211'.
  warnings.warn(
100%|██████████| 100/100 [01:04<00:00,  1.55it/s]


... (7 outputs left out)

/home/akka/Documents/vscode_projects/SemesterProjectFall2026/.venv/lib/python3.12/site-packages/sdv/single_table/base.py:136: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
/home/akka/Documents/vscode_projects/SemesterProjectFall2026/.venv/lib/python3.12/site-packages/sdv/metadata/metadata.py:1226: UserWarning: The metadata lists columns in a different order than the data. This may result in the synthetic data having a different order.
Affected tables: 'sinetänsalmentie_Acc1_m794_RDMC_2026-05-06_112959'.
  warnings.warn(
100%|██████████| 100/100 [01:03<00:00,  1.57it/s]


Saved ./data_features/synth/sinetänsalmentie_Acc1_m794_RDMC_2026-05-06_112959+.csv

=== sinetänsalmentie_Acc1_m795_RDMC_2026-05-06_113156 ===
Created 361 windows of 50 rows each (18050 total rows).


/home/akka/Documents/vscode_projects/SemesterProjectFall2026/.venv/lib/python3.12/site-packages/sdv/single_table/base.py:136: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
/home/akka/Documents/vscode_projects/SemesterProjectFall2026/.venv/lib/python3.12/site-packages/sdv/metadata/metadata.py:1226: UserWarning: The metadata lists columns in a different order than the data. This may result in the synthetic data having a different order.
Affected tables: 'sinetänsalmentie_Acc1_m795_RDMC_2026-05-06_113156'.
  warnings.warn(
100%|██████████| 100/100 [01:03<00:00,  1.57it/s]


Saved ./data_features/synth/sinetänsalmentie_Acc1_m795_RDMC_2026-05-06_113156+.csv

Done!
